#1. Basic Tasks 

### 1.Create a notebook and add a markdown cell summarizing, in your own words, the difference between a warehouse, a lake, and a lakehouse. 

##Data warehouse:
-Data warehouse is like storege data where stuctured data(like tables,csv,json, parquet) is stored but has some storage constraints.

-for analogy data warehouse is like water storage tank for residency it can store some amount of data not BigData.

##Data lake:
-Data lake can handle both structured and unstructured data(videos,images,pdf etc.).


-for analogy as name suggested it is like lake where you can any kind of water and it kind of vast as compared to data warehouse.


-on all above it maintain ACID property. 

##Data lakehouse:
-the main diff between data lake and data lake house mostly like datalakehouse manage ACID property well than data lake.


-it have property of both data warehouse and data lake.

### 2. Create a Delta table from ~10 rows of sample product data (product_id, name, category, price). 

In [0]:
%sql

--creating delta table with table name assign_prod in dev catlog and bronze schema
create or replace table dev.bronze.assign_prod(product_id int,product_name string,product_category string,product_price double);

--hard code data into table assign_prod
insert into dev.bronze.assign_prod values(1,'product1','category1',100),(2,'product2','category2',200),(3,'product3','category3',300),(4,'product4','category4',400),(5,'product5','category5',500),(6,'product6','category6',600),(7,'product7','category7',700),(8,'product8','category8',800),(9,'product9','category9',900),(10,'product10','category10',1000);

##3. Run three separate INSERT/UPDATE statements against the table, then use DESCRIBE HISTORY to view the resulting versions. 


In [0]:
--inserting and updating data in assign_prod
insert into dev.bronze.assign_prod values(11,'product11','category11',1100);
update dev.bronze.assign_prod set product_price=product_price+100 where product_id=11;
update dev.bronze.assign_prod set product_category='category10' where product_id=11;

describe history dev.bronze.assign_prod;

##4. (Data Analyst) Use SELECT ... VERSION AS OF to query an older version of the table and note what changed between versions. 

In [0]:
--view versioning of delta table dev.bronze.assign_prod
select * from dev.bronze.assign_prod version AS of 10;


In [0]:
select * from dev.bronze.assign_prod version AS of 12;

##Difference between two version
The main difference between version 10 and 12 
- in version 10:
the product11 has product_category as category11 and price 1100

- in version 12:
the product11 has product_category as category10 and price 1200 

#2. Intermediate Tasks 

##

##5. Deliberately insert a row with an extra column and observe Delta's schema enforcement rejecting it; then re-insert using mergeSchema and confirm schema evolution succeeded. 

In [0]:
insert into dev.bronze.assign_prod values (12,'product12','category12',1200,2026);


In [0]:
-- Using schema evolution in SQL to add a new column named 'year'
INSERT WITH SCHEMA EVOLUTION INTO dev.bronze.assign_prod 
SELECT 13 AS product_id, 'product13' AS product_name, 'category13' AS product_category, 1300 AS product_price, 2027 AS year;

##6. Use time travel (VERSION AS OF and TIMESTAMP AS OF) to reconstruct the table as it looked before a simulated bad update, then write the RESTORE command that would fix it. 

In [0]:
select * from dev.bronze.assign_prod version as of 12;

In [0]:
select * from dev.bronze.assign_prod timestamp as of '2026-08-20T11:36:13.000+00:00';

In [0]:
-- RESTORE to version 12 (before the schema evolution in Cell 14)
restore dev.bronze.assign_prod to version as of 14

##7. Write a short explanation, aimed at a non-technical stakeholder, of why ACID transactions matter when multiple pipelines write to the same table concurrently. 


ACID transactions ensure that when multiple data pipelines write to the same table at the same time, the table stays accurate and reliable. They prevent issues like partial updates, conflicting changes, or corrupted data, so everyone sees consistent information and business decisions are based on trustworthy data.


#3. Advanced Tasks 

##8. Write a short design note describing how Cyntexa could replace a nightly batch warehouse load with a lakehouse pipeline, calling out specifically where ACID transactions and time travel reduce operational risk compared to the current warehouse. 

## Design Note: Migrating from Nightly Batch Warehouse to Lakehouse Pipeline

### Current Warehouse Pain Points
* Nightly batch window creates 12-24 hour data latency
* Failed loads require full reprocessing (hours of downtime)
* Concurrent writes need complex locking and coordination
* No easy rollback—bad load means drop table and reload from scratch
* Limited audit trail for debugging and compliance

### Proposed Lakehouse Architecture
* **Replace batch with incremental pipeline**: Stream or micro-batch ingestion throughout the day
* **Delta Lake as unified storage**: ACID guarantees + time travel built-in
* **Real-time analytics**: Business decisions on current data, not yesterday's snapshot

---

## Where ACID Transactions Reduce Risk

### Problem: Data Corruption from Failed/Concurrent Writes
**Traditional Warehouse:**
* Batch fails halfway → table left with partial/corrupted data
* Multiple pipelines writing → need explicit locks and scheduling windows
* Rollback requires manual cleanup scripts

**Lakehouse Solution (ACID):**
* **Atomicity**: Write succeeds completely or not at all—no partial updates visible
* **Consistency**: Multiple teams write concurrently without table corruption
* **Isolation**: Readers always see complete, consistent snapshots during writes
* **Durability**: Committed data survives failures automatically

**Operational Impact:**
* ✓ Zero "broken batch" incidents
* ✓ No coordination windows—teams work independently
* ✓ Automatic consistency without custom locking logic

---

## Where Time Travel Reduces Risk

### Problem: Slow, Manual Recovery from Bad Data
**Traditional Warehouse:**
* Bad load detected → drop table + reload from source (hours)
* "What changed?" requires comparing backup snapshots
* Root cause analysis = manual log parsing

**Lakehouse Solution (Time Travel):**
* **Instant rollback**: `RESTORE TABLE TO VERSION AS OF 10` (seconds, not hours)
* **Built-in audit**: `DESCRIBE HISTORY` shows every change with timestamp, user, operation
* **Point-in-time comparison**: `SELECT ... VERSION AS OF 9` vs. current to see exact differences
* **Zero data loss**: All versions retained per policy—forensics always possible

**Operational Impact:**
* ✓ Recovery time: Hours → Seconds
* ✓ Automatic compliance audit trail
* ✓ "Time travel debugging"—reproduce exact state when issue occurred

---

## Implementation Pointers for Cyntexa

**Phase 1: Pilot (Week 1-2)**
* Pick one high-value table (e.g., customer transactions)
* Convert batch INSERT → incremental MERGE on Delta Lake
* Establish alerting on pipeline health

**Phase 2: Scale (Week 3-6)**
* Migrate critical business tables
* Set schema evolution policies (`mergeSchema` for approved changes)
* Train team: `RESTORE`, `VERSION AS OF`, `DESCRIBE HISTORY`

**Phase 3: Production (Week 7+)**
* Migrate remaining tables
* Sunset nightly batch jobs
* Monitor cost and performance optimization

---

##9. Simulate two concurrent writers appending to the same Delta table (two notebook cells or jobs), then use DESCRIBE HISTORY to explain how the transaction log resolved the write order and what would happen if the writes conflicted. 

##9. Simulating Concurrent Writers and Transaction Log Resolution

### Step 1: Two Concurrent Appends

- **Writer 1 (Job/Notebook A):**
  
  insert into dev.bronze.assign_prod values (14,'product14','category14',1400);
  
- **Writer 2 (Job/Notebook B):**
  
  insert into dev.bronze.assign_prod values (15,'product15','category15',1500);
  

> These could run at nearly the same time.

---

### Step 2: DESCRIBE HISTORY to Trace the Transaction Log


describe history dev.bronze.assign_prod;


*Each write becomes a new version in the Delta transaction log, with timestamp, user, operation and job id.*

---

### Step 3: Explanation

- **How Delta Resolves Write Order:**
  - Delta automatically serializes transactions. Even though both writers may run "at the same time," the Delta log applies one, then the other. Each write gets its own unique version (e.g., version 15, then 16).
  - The **DESCRIBE HISTORY** output shows each append, their order, timestamp, and job/user.

- **If the Writes Conflict:**
  - If both writers try to update the same row, Delta's conflict detection aborts one job and retries automatically, or surfaces an error. Only one write is committed per version—never partial or overlapping changes.

---

**Result:**  
- Data is consistent: no partial data or lost updates.  
- The DESCRIBE HISTORY log makes it easy to audit the order and outcome of concurrent writes.

In [0]:
describe history dev.bronze.assign_prod;

##10. (Data Analyst) Write a one-page comparison memo: list 3 concrete advantages a lakehouse gives analysts over a traditional warehouse, and 1 tradeoff to watch for. 

## Lakehouse vs. Traditional Warehouse: Analyst Perspective

### Lakehouse Advantages

1. **Real-Time Data Access**
   - Lakehouses allow analysts to query up-to-the-minute data, supporting streaming and micro-batch ingestion, while traditional warehouses often impose hours-long latency due to nightly batch loads.

2. **Easy Time Travel & Rollbacks**
   - Delta Lakehouse technology lets analysts instantly view and revert to previous table versions (via `VERSION AS OF` or `RESTORE`), enabling rapid error correction, historical analysis, and auditability—something that requires complex processes in warehouses.

3. **Seamless Schema Evolution**
   - Lakehouse platforms support schema changes (e.g., automatically adding columns with `MERGE SCHEMA`) without downtime or manual migration, letting analysts adapt datasets to evolving business requirements, unlike rigid warehouse schemas.

### Tradeoff to Watch For

1. **Performance Variability**
   - While lakehouses scale efficiently and handle diverse workloads, complex queries on massive raw data (especially unoptimized tables) can run slower than highly-tuned warehouse systems. Analysts need to monitor query performance and leverage optimization features (e.g., caching)